In [10]:
# Physics Constants (Explicit Float32)
const Cm  = 1.0f0      # µF/cm²
const g_Na = 120.0f0    # mS/cm²
const E_Na = 50.0f0     # mV
const g_L  = 0.3f0      # mS/cm²
const E_L  = -54.387f0  # mV
const g_K  = 36.0f0    #potassium conductance (mS/cm²)
const E_K  = -77.0f0 
# Voltage-gated ion channel kinetics (Float32)

α_n(V::Float32) = 0.01f0 * (V + 55f0) / (1f0 - exp(-(V + 55f0) / 10f0))
β_n(V::Float32) = 0.125f0 * exp(-(V + 65f0) / 80f0)

α_m(V::Float32) = 0.1f0 * (V + 40f0) / (1f0 - exp(-(V + 40f0) / 10f0))
β_m(V::Float32) = 4.0f0 * exp(-(V + 65f0) / 18f0)

α_h(V::Float32) = 0.07f0 * exp(-(V + 65f0) / 20f0)
β_h(V::Float32) = 1f0 / (1f0 + exp(-(V + 35f0) / 10f0))

# Steady-state & time-constant functions (Float32)

m_inf(V::Float32) = α_m(V) / (α_m(V) + β_m(V))
h_inf(V::Float32) = α_h(V) / (α_h(V) + β_h(V))
n_inf(V::Float32) = α_n(V) / (α_n(V) + β_n(V))

tau_n(V::Float32) = 1f0 / (α_n(V) + β_n(V))

println("Physics of neural dynamics defined in Float32")


Physics of neural dynamics defined in Float32


In [ ]:
using SciMLSensitivity, DifferentialEquations, ModelingToolkit, Optimization, OptimizationOptimisers
using Lux, Random, ComponentArrays, CUDA, Zygote
using Plots, JLD2

# 1. SETUP & DATA (Move to GPU)
# ------------------------------------------------
@load "Data/synthetic_data/noise_0_hh_2d_model.jld" V
t_train = 0.0f0:0.1f0:50.0f0

# Move data to GPU
V_gpu = cu(Float32.(V))
t_train_gpu = cu(t_train)
# Also keep a CPU copy of V for safe CPU-side loss computation
V_cpu = Array(V_gpu)

# 2. DEFINE MODEL WITH LUX (The Modern Way)
# ------------------------------------------------
# Lux models are explicit. We don't 'destructure' them.
# We define the structure once.
nn = Lux.Chain(
    Lux.Dense(1, 16, tanh),
    Lux.Dense(16, 1)
)

# Initialize parameters (p) and state (st)
rng = Random.default_rng()
p_init, st = Lux.setup(rng, nn)

# CRITICAL STABILITY FIX
# Set final layer weights to zero to prevent initial explosion
p_init.layer_2.weight .= 0.0f0
p_init.layer_2.bias   .= 0.0f0

# Move parameters to GPU
p_gpu = cu(ComponentArray(p_init))
st_gpu = cu(st) # State also needs to be on GPU

# External stimulus function (simple square pulse). Replace with your experiment's stimulus as needed.
function Stimulus(t)
    return (t >= 10.0f0 && t < 11.0f0) ? 20.0f0 : 0.0f0
end

# 3. UDE FUNCTION (Optimized for GPU)
# ------------------------------------------------
# We pass the Lux model structure 'nn' and state 'st' via a closure or global
function hodgkin_huxley_UDE!(du, u, p, t, nn, st)
    # Avoid scalar indexing on a CuArray `u` (disallowed on GPU).
    # Copy the small state vector to CPU and work with it instead.
    u_local = Array(u)
    V, n = u_local
    
    # LUX FORWARD PASS
    # Run the NN on GPU (parameters `p`/`st` are on GPU) but keep physics on CPU scalars.
    _V_input_gpu = cu(Float32[V])
    pred_gpu, _ = nn(_V_input_gpu, p, st)
    pred_cpu = Array(pred_gpu)
    pred_I_Na = pred_cpu[1]
    
    # Physics calculations on CPU scalars
    I_ext = Stimulus(t)
    I_K = g_K * n^4 * (V - E_K)
    I_L = g_L * (V - E_L)
    du_local1 = (I_ext - (pred_I_Na + I_K + I_L) / Cm)
    du_local2 = (n_inf(V) - n) / tau_n(V)
    
    # Transfer derivatives back to GPU `du`
    du .= cu(Float32[du_local1, du_local2])
end

# 4. WRAPPER FOR SCIML
# ------------------------------------------------
# DifferentialEquations expects (du, u, p, t)
# We wrap the Lux model and state into the function
ude_dynamics!(du, u, p, t) = hodgkin_huxley_UDE!(du, u, p, t, nn, st_gpu)

u0_gpu = cu([-65.0f0, 0.317f0])
prob_nn = ODEProblem(ude_dynamics!, u0_gpu, (0.0f0, 50.0f0), p_gpu)

# 5. SOLVE (Using a GPU-friendly solver)
# ------------------------------------------------
function loss(p)
    # GBS (Grossmann-Bulirsch-Stoer) is often robust for simple problems
    # Tsit5 is standard. 
    pred = solve(prob_nn, Tsit5(), p=p, saveat=t_train_gpu, 
                 reltol=1e-4, abstol=1e-4) # Lower tolerance for Float32
    
    if pred.retcode != :Success
        return 1f6
    end
    
    # Vectorized GPU loss calculation (compute on CPU to avoid scalar GPU indexing)
    pred_arr = Array(pred)  # materialize solution on CPU
    loss_val = sum(abs2, pred_arr[1,:] .- V_cpu)
    return loss_val
end

# ... Optimization follows ...
optf = Optimization.OptimizationFunction((x, p) -> loss(x), Optimization.AutoZygote())
optprob = Optimization.OptimizationProblem(optf, p_gpu) # Pass GPU parameters

res = Optimization.solve(optprob, OptimizationOptimisers.Adam(0.01), maxiters=1000)

GPUCompiler.KernelError: GPU compilation of MethodInstance for (::GPUArrays.var"#gpu_broadcast_kernel_linear#39")(::KernelAbstractions.CompilerMetadata{KernelAbstractions.NDIteration.DynamicSize, KernelAbstractions.NDIteration.DynamicCheck, Nothing, CartesianIndices{1, Tuple{Base.OneTo{Int64}}}, KernelAbstractions.NDIteration.NDRange{1, KernelAbstractions.NDIteration.DynamicSize, KernelAbstractions.NDIteration.DynamicSize, CartesianIndices{1, Tuple{Base.OneTo{Int64}}}, CartesianIndices{1, Tuple{Base.OneTo{Int64}}}}}, ::CuDeviceVector{Float32, 1}, ::Base.Broadcast.Broadcasted{CUDA.CuArrayStyle{1, CUDA.DeviceMemory}, Tuple{Base.OneTo{Int64}}, typeof(-), Tuple{Base.Broadcast.Extruded{CuDeviceVector{Float32, 1}, Tuple{Bool}, Tuple{Int64}}, Base.Broadcast.Extruded{Vector{Float32}, Tuple{Bool}, Tuple{Int64}}}}) failed
KernelError: passing non-bitstype argument

Argument 4 to your kernel function is of type Base.Broadcast.Broadcasted{CUDA.CuArrayStyle{1, CUDA.DeviceMemory}, Tuple{Base.OneTo{I